# 03 — Visualizations (Deliverable C, figures 1-9)

Figures 1-9 only need the cleaned data and are built here. Figures 10-12
(model comparison, residuals, feature importance) need the trained model and
are generated at the end of notebook 04 instead, then this same captions file
is updated there. See the note in the last cell.

In [ ]:
import sys, os
sys.path.append(os.path.join('..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from features import attach_price

RAW = os.path.join('..', 'data', 'raw')
PROCESSED = os.path.join('..', 'data', 'processed')
ASSETS = os.path.join('..', 'app', 'assets')
FIGURES = os.path.join('..', 'figures')
os.makedirs(FIGURES, exist_ok=True)

raw = pd.read_csv(os.path.join(RAW, 'crop_yield_train.csv'))
master_train = pd.read_csv(os.path.join(PROCESSED, 'master_train.csv'))
price_clean = pd.read_csv(os.path.join(ASSETS, 'price_clean.csv'))
weather_clean = pd.read_csv(os.path.join(ASSETS, 'weather_clean.csv'))

print(raw.shape, master_train.shape, price_clean.shape, weather_clean.shape)

## Figure 1 — Missingness by column (raw training data)

In [ ]:
missing = raw.isna() | raw.isin([-999, -999.0])
miss_pct = missing.mean().sort_values(ascending=False) * 100
miss_pct = miss_pct[miss_pct > 0]

fig, ax = plt.subplots(figsize=(7, 4))
miss_pct.plot(kind='bar', ax=ax, color='#937860')
ax.set_ylabel('Percent missing (including -999 sentinels)')
ax.set_title('Fig 1: Missingness by column, raw training data')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig01_missingness.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — fertilizer_kg_per_ha before vs after cleaning

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].hist(raw['fertilizer_kg_per_ha'].dropna(), bins=40, color='#4C72B0')
axes[0].set_title('Before cleaning')
axes[0].set_xlabel('fertilizer_kg_per_ha')
axes[1].hist(master_train['fertilizer_kg_per_ha'], bins=40, color='#55A868')
axes[1].set_title('After cleaning (imputed + capped)')
axes[1].set_xlabel('fertilizer_kg_per_ha')
fig.suptitle('Fig 2: fertilizer_kg_per_ha before vs after cleaning')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig02_before_after_cleaning.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — Yield distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(master_train['yield_tons_per_ha'], bins=40, color='#8172B2')
ax.set_xlabel('Yield (tons/ha)')
ax.set_ylabel('Count')
ax.set_title('Fig 3: Distribution of yield_tons_per_ha')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig03_yield_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 4 — Region x crop yield heatmap

In [ ]:
pivot = master_train.pivot_table(index='region', columns='crop_type', values='yield_tons_per_ha', aggfunc='mean')

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(pivot.values, cmap='YlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f'{pivot.values[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, label='Mean yield (tons/ha)')
ax.set_title('Fig 4: Mean yield by region and crop')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig04_region_crop_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 5 — Feature correlation heatmap

In [ ]:
numeric_cols = [
    'altitude_m', 'rainfall_mm_season', 'farm_size_ha', 'fertilizer_kg_per_ha',
    'improved_seed_used', 'pest_disease_flag', 'soil_quality_index', 'labor_days_per_ha',
    'distance_to_market_km', 'season_avg_temp_c', 'season_rainfall_mm_total',
    'season_extreme_heat_days', 'temp_deviation_from_region_avg',
    'fertilizer_x_improved_seed', 'planting_month_num', 'yield_tons_per_ha',
]
corr = master_train[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=90)
ax.set_yticks(range(len(numeric_cols)))
ax.set_yticklabels(numeric_cols)
plt.colorbar(im, ax=ax, label='Correlation')
ax.set_title('Fig 5: Feature correlation heatmap')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig05_correlation_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 6 — Climate by region

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
weather_clean.boxplot(column='avg_temp_c', by='region', ax=axes[0])
axes[0].set_title('Avg temp by region')
axes[0].set_xlabel('')
weather_clean.boxplot(column='monthly_rainfall_mm', by='region', ax=axes[1])
axes[1].set_title('Monthly rainfall by region')
axes[1].set_xlabel('')
fig.suptitle('Fig 6: Climate by region')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig06_climate_by_region.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 7 — Yield vs season average temperature

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(master_train['season_avg_temp_c'], master_train['yield_tons_per_ha'], alpha=0.2, s=8)
ax.set_xlabel('Season average temperature (C)')
ax.set_ylabel('Yield (tons/ha)')
ax.set_title('Fig 7: Yield vs season average temperature')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig07_yield_vs_season_temp.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 8 — Crop price trends

In [ ]:
price_pivot = price_clean.groupby(['year', 'crop_type'])['price_birr_per_quintal'].mean().unstack()

fig, ax = plt.subplots(figsize=(7, 4))
price_pivot.plot(ax=ax, marker='o')
ax.set_ylabel('Price (birr/quintal)')
ax.set_title('Fig 8: Crop price trends, 2021-2024')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig08_price_trends.png'), dpi=150, bbox_inches='tight')
plt.show()

## Figure 9 — Revenue per hectare by crop and region

In [ ]:
revenue_df = attach_price(master_train, price_clean)
rev_pivot = revenue_df.pivot_table(index='region', columns='crop_type', values='revenue_birr_per_ha', aggfunc='mean')

fig, ax = plt.subplots(figsize=(8, 5))
rev_pivot.plot(kind='bar', ax=ax)
ax.set_ylabel('Revenue (birr/ha)')
ax.set_title('Fig 9: Mean revenue per hectare by crop and region')
plt.legend(title='crop_type', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES, 'fig09_revenue_by_crop_region.png'), dpi=150, bbox_inches='tight')
plt.show()

## Captions for figures 1-9

Figures 10-12 are added by notebook 04 once the model exists; this cell only
touches the 9 lines it owns and leaves the rest of the file untouched.

In [ ]:
captions_path = os.path.join(FIGURES, 'figure_captions.md')

new_captions = {
    'fig01_missingness.png': (
        f"Shows percent missing (including -999 sentinels) per raw column; "
        f"{', '.join(miss_pct.index[:3])} are the most affected, which is why they get "
        f"train-fit median/mode imputation in deliverable A."
    ),
    'fig02_before_after_cleaning.png': (
        "Compares fertilizer_kg_per_ha before and after cleaning; capping at the train "
        "99th percentile removes the extreme right tail without reshaping the bulk of "
        "the distribution."
    ),
    'fig03_yield_distribution.png': (
        f"Yield is right-skewed with a mean of {master_train['yield_tons_per_ha'].mean():.2f} "
        f"tons/ha; most plots sit below a smaller number of high-yield outliers."
    ),
    'fig04_region_crop_heatmap.png': (
        "Mean yield varies more by crop than by region; the darkest cell marks the single "
        "best-performing region-crop combination."
    ),
    'fig05_correlation_heatmap.png': (
        "No single feature correlates strongly with yield on its own, which is the "
        "motivation for using a non-linear model rather than plain linear regression in D."
    ),
    'fig06_climate_by_region.png': (
        "Temperature and rainfall both vary noticeably by region, which justifies doing "
        "region-aware (not just national) weather aggregation."
    ),
    'fig07_yield_vs_season_temp.png': (
        "Yield shows only a loose relationship with season average temperature, visible as "
        "a broad cloud rather than a tight line."
    ),
    'fig08_price_trends.png': (
        "Crop prices generally trended upward from 2021 to 2024, with teff consistently the "
        "most expensive crop per quintal."
    ),
    'fig09_revenue_by_crop_region.png': (
        "Revenue per hectare reflects both yield and price, so the highest-yield crop is not "
        "always the highest-revenue crop once price is factored in."
    ),
}

with open(captions_path) as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    for fig_name, caption in new_captions.items():
        if fig_name in line:
            lines[i] = f"- **{fig_name}:** {caption}\n"

with open(captions_path, 'w') as f:
    f.writelines(lines)

print('Updated captions for fig01-fig09. fig10-fig12 are filled in by notebook 04.')